# **1. Setup & Data Loading**
## Coast-to-Coast Travel Analysis

This notebook includes:
- Package installation and imports
- Loading NFL schedule data (2021-2024)
- Loading team performance statistics
     

---
## **Import Libraries**

In [34]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # To ignore warnings for cleaner output

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid') # This line sets the style for matplotlib plots, dark background with grid lines
sns.set_palette("husl") # Color palette to shwo differences in data, colorful and vibrant 


---
## **Load Schedule Data**

The schedule data contains:
- Game dates and times
- Home and away teams
- Game results
- Stadium information

In [35]:
!pip install pyarrow --upgrade
# This package is required to read the nflfastR data files, which are in Apache Arrow format



[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [36]:
# Load team-level stats
team_stats = nfl.load_team_stats([2021, 2022, 2023, 2024])
team_stats = team_stats.to_pandas()  # Convert from Polars to pandas for easier manipulation

# Load schedules
schedules = nfl.load_schedules([2021, 2022, 2023, 2024])
schedules = schedules.to_pandas()

---
## **Schedule Data**

In [37]:
# Check the structure
print("Dataset shape:", schedules.shape)
print("\nColumn names:")
print(list(schedules.columns))

Dataset shape: (1139, 46)

Column names:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [38]:
# View first few rows
schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2021_01_DAL_TB,2021,REG,1,2021-09-09,Thursday,20:20,DAL,29,TB,...,9.0,00-0033077,00-0019596,Dak Prescott,Tom Brady,Mike McCarthy,Bruce Arians,Shawn Hochuli,TAM00,Raymond James Stadium
1,2021_01_PHI_ATL,2021,REG,1,2021-09-12,Sunday,13:00,PHI,32,ATL,...,NaN,00-0036389,00-0026143,Jalen Hurts,Matt Ryan,Nick Sirianni,Arthur Smith,Scott Novak,ATL97,Mercedes-Benz Stadium
2,2021_01_PIT_BUF,2021,REG,1,2021-09-12,Sunday,13:00,PIT,23,BUF,...,17.0,00-0022924,00-0034857,Ben Roethlisberger,Josh Allen,Mike Tomlin,Sean McDermott,John Hussey,BUF00,New Era Field
3,2021_01_NYJ_CAR,2021,REG,1,2021-09-12,Sunday,13:00,NYJ,14,CAR,...,0.0,00-0037013,00-0034869,Zach Wilson,Sam Darnold,Robert Saleh,Matt Rhule,Clay Martin,CAR00,Bank of America Stadium
4,2021_01_MIN_CIN,2021,REG,1,2021-09-12,Sunday,13:00,MIN,24,CIN,...,15.0,00-0029604,00-0036442,Kirk Cousins,Joe Burrow,Mike Zimmer,Zac Taylor,Adrian Hill,CIN00,Paul Brown Stadium


In [39]:
# Define key columns
key_cols = ['gameday', 'home_team', 'away_team', 'home_score', 'away_score']

# Check for missing values in key columns
print("Missing values:")
print(schedules[key_cols].isna().sum())

Missing values:
gameday       0
home_team     0
away_team     0
home_score    0
away_score    0
dtype: int64


---
## **Load Team Statistics**

Weekly team statistics include:
- Points scored and allowed
- Offensive yards (passing, rushing)
- Turnovers
- Third down conversions
- Time of possession

In [40]:
# Define years to analyze
years = [2021, 2022, 2023, 2024]

# Load team-level stats
team_stats = nfl.load_team_stats(years)
team_stats = team_stats.to_pandas()


---
## **Explore Team Statistics**

In [41]:
# Check structure
print("Dataset shape:", team_stats.shape)
print("\nColumn names:")
print(list(team_stats.columns))

Dataset shape: (2278, 102)

Column names:
['season', 'week', 'team', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tackles_for_loss', 'def_tackles_for_loss_yards', 'def_fumbles_forced', 'def_sacks', 'def_sack_yards', 'def_qb_hits', 'def_intercept

In [42]:
# View the fist 5 lines of the dataset
team_stats.head()

,season,week,team,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,...,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance
0,2021,1,ARI,REG,TEN,21,32,289,4,1,...,5,5,0,0,1.0,0,0,0,0,0
1,2021,1,ATL,REG,PHI,21,35,164,0,0,...,0,0,0,0,NaN,0,0,0,0,0
2,2021,1,BAL,REG,LV,19,30,235,1,0,...,3,3,0,0,1.0,0,0,0,0,0
3,2021,1,BUF,REG,PIT,30,51,270,1,0,...,1,1,0,0,1.0,0,0,0,0,0
4,2021,1,CAR,REG,NYJ,24,35,279,1,0,...,1,2,1,0,0.5,0,0,0,0,0


In [43]:
# Identify any key performance metrics using key words
performance_cols = [col for col in team_stats.columns if any(keyword in col.lower() 
                    for keyword in ['score', 'yards', 'pass', 'rush', 'turnover', 'points'])]

print(f"{len(performance_cols)} performance-related columns:")
for col in performance_cols[:20]:  # Showing first 20 as example
    print(f"  - {col}")

30 performance-related columns:
  - passing_yards
  - passing_tds
  - passing_interceptions
  - sack_yards_lost
  - passing_air_yards
  - passing_yards_after_catch
  - passing_first_downs
  - passing_epa
  - passing_cpoe
  - passing_2pt_conversions
  - rushing_yards
  - rushing_tds
  - rushing_fumbles
  - rushing_fumbles_lost
  - rushing_first_downs
  - rushing_epa
  - rushing_2pt_conversions
  - receiving_yards
  - receiving_air_yards
  - receiving_yards_after_catch


---
## Save the loaded data for use in subsequent notebooks

In [44]:
# Save to CSV for easy loading later
schedules.to_csv('data_schedules.csv', index=False)
team_stats.to_csv('data_team_stats.csv', index=False)
